# Cuestiones metodológicas: clasificación, periodos e inferencia
**Pablo Sánchez González · Universidad Complutense de Madrid · Economía Pública**

## Resumen de decisiones
1. Los 240 huecos en PC_GDP se convierten en ceros **solo** al corresponder a subsectores sin registro separado documentado: 230 S.1312 y 10 S.1314 de Malta.
2. Los trienios 2017–2019 y 2022–2024 estiman una diferencia de composición media entre etapas. La serie anual se conserva para examinar dinámica, tendencia previa y evolución posterior.
3. La inferencia es exploratoria y secundaria a la descripción; ni periodizar ni añadir efectos temporales crea un contrafactual.

## 1. Preparación y trazabilidad
Los datos se decodifican desde la instantánea guardada. El tratamiento conserva `value`, `status`, `analysis_value`, `treatment` y `source_rule`, evitando confundir una decisión analítica con un valor publicado.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src/analysis.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis.py").exists(), "Ejecutar desde la raíz o notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from src.prepare_data import decode_snapshot, prepare, SHARES, STATE_COUNTRIES
from src.analysis import (
    LABELS,
    NAMES,
    changes,
    inference,
    period_mean,
    temporal_sensitivity,
    leave_one_out,
    grouped_change,
)

raw, snapshot = decode_snapshot()
wide, audit = prepare(raw)
pd.set_option("display.max_columns", 12)

## 2. Qué significa un cero institucional
En esta investigación, un cero indica que un subsector no presenta gasto por separado bajo la clasificación utilizada. No indica ausencia de la función económica correspondiente.

El [glosario de Eurostat](https://ec.europa.eu/eurostat/statistics-explained/SEPDF/cache/1123.pdf), p. 1, identifica S.1312 en Bélgica, Alemania, España y Austria dentro de la UE. El [inventario EDP de Malta](https://nso.gov.mt/wp-content/uploads/2023/01/EDP-Inventory-ESA-2010.pdf), apartados 1.2 y 1.4, documenta que allí S.1312 y S.1314 no son aplicables; el gasto social se registra dentro del gobierno central. Fuentes consultadas el 24-09-2026.

La aplicación a 2015–2024 combina esa evidencia institucional con el patrón uniforme de la instantánea. La marca `m` por sí sola no justifica una imputación. Una nueva ausencia fuera de esta lista provoca un error, no un cero.

In [2]:
missing = audit[audit["value"].isna()]
zero_map = missing.groupby(["geo", "sector"]).agg(
    observaciones=("time", "size"),
    primer_ano=("time", "min"),
    ultimo_ano=("time", "max"),
    regla=("source_rule", "first"),
)
display(zero_map)
assert len(missing) == 240
assert set(missing.query("sector == 'S1314'").geo) == {"MT"}
assert not set(missing.query("sector == 'S1312'").geo) & STATE_COUNTRIES
assert missing.analysis_value.eq(0).all()

,,observaciones,primer_ano,ultimo_ano,regla
geo,sector,,,,
BG,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
CY,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
CZ,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
DK,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
EE,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
EL,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
FI,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
FR,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...
HR,S1312,10,2015,2024,https://ec.europa.eu/eurostat/statistics-expla...


### Por qué no eliminar países ni rellenar todos los huecos
Exigir cuatro subsectores observados dejaría únicamente AT, BE, DE y ES: responderíamos una pregunta sobre esos cuatro países, no sobre la UE-27. Imputar universalmente cero ocultaría falta de información. La lista cerrada evita ambos problemas.

Tampoco se puede usar una transformación logarítmica ordinaria de cuatro componentes con ceros estructurales sin modificar el objeto estudiado. No se añaden pseudocuentas arbitrarias. Para regional se ofrece descripción y se evita una prueba sustentada en 23 ceros.

In [3]:
counts = (
    audit[audit["sector"].isin(["S1311", "S1312", "S1313", "S1314"])]
    .groupby("geo")
    .value.count()
)
print(
    "Países con cuatro subsectores numéricos cada año:",
    list(counts[counts.eq(40)].index),
)
# Prueba de la regla: simular un dato central desconocido debe detenerse.
broken = raw.copy()
mask = (
    (broken.unit == "PC_GDP")
    & (broken.geo == "ES")
    & (broken.time == 2019)
    & (broken.sector == "S1311")
)
broken.loc[mask, "value"] = np.nan
try:
    prepare(broken)
except ValueError as error:
    print("Control esperado:", str(error)[:140])
else:
    raise AssertionError("Una ausencia no documentada fue aceptada")

Países con cuatro subsectores numéricos cada año: ['AT', 'BE', 'DE', 'ES']
Control esperado: Ausencia no documentada: [{"geo":"ES","time":2019,"sector":"S1311"}]


## 3. Denominador y significado económico
Se normaliza por la suma del gasto de cuatro subsectores. Según [Eurostat, apartado 18.5](https://webgate.ec.europa.eu/eurostat/cache/metadata/en/gov_10a_exp_esms.htm), el total S.13 consolida determinadas operaciones internas; por eso no coincide con dicha suma. Dividir cada subsector por S.13 no produciría cuotas que sumen 100.

Ejemplo: una transferencia central de 10 que financia gasto local de 10 puede aparecer en ambos niveles antes de consolidar. Las cuotas informan sobre registro y financiación institucional, no sobre el reparto exclusivo del gasto final. El PIB común se cancela algebraicamente; la comprobación en euros mide el efecto del redondeo previo.

In [4]:
gap = wide["subsector_total"] - wide.S13
display(gap.describe().to_frame("D menos S13 (pp PIB)").round(3))
wide_eur, _ = prepare(raw, "MIO_EUR")
display(
    pd.DataFrame(
        {
            "PC_GDP": changes(wide)[SHARES].mean(),
            "MIO_EUR": changes(wide_eur)[SHARES].mean(),
        }
    ).round(6)
)

,D menos S13 (pp PIB)
count,270.000
mean,11.025
std,5.534
min,0.200
25%,7.725
50%,11.450
75%,13.775
max,25.300


,PC_GDP,MIO_EUR
share_central,1.148329,1.147844
share_state,-0.023869,-0.024966
share_local,-0.456123,-0.445082
share_social_security,-0.668337,-0.677796


## 4. Por qué comparar periodos
La pregunta sustantiva es si una etapa posterior conserva una composición distinta a la etapa anterior, no estimar cada fluctuación anual. Se mantiene la elección original de 2017–2019 como base reciente y 2022–2024 como etapa posterior a la fase aguda. **No es una selección prerregistrada**, y 2022–2024 no se presenta como periodo libre de perturbaciones.

Cada media de tres años amortigua valores anuales singulares y mantiene igual peso por país y año. La diferencia emparejada elimina de la comparación los niveles nacionales constantes, pero no sus tendencias. No equivale a la cuota del gasto acumulado: esa razón ponderaría años según el gasto total.

### Qué se pierde y cómo se compensa
La agregación oculta dinámica interna y no separa el salto de 2020 de su reversión posterior. Por eso se conserva la serie de diez años y se comparan explícitamente la fase aguda y la posterior. Bajo autocorrelación positiva, promediar no reduce la varianza como si los años fueran independientes: para tres observaciones, $\mathrm{Var}(\bar u)=\{3\gamma_0+4\gamma_1+2\gamma_2\}/9$.

### Por qué no una regresión temporal como resultado principal
Diez puntos anuales y cinco anteriores a 2020 ofrecen poca información para distinguir tendencia, ruptura y persistencia. Un panel puede aprovechar trayectorias nacionales, pero 270 filas no son 270 realizaciones independientes de la pandemia. La exposición temporal es común: con efectos fijos de año, un indicador post común es colineal; sin ellos, recoge también shocks simultáneos. La agregación es una elección de estimando y transparencia, no una demostración de que una serie temporal sea inválida.

In [5]:
annual = wide.groupby("time")[SHARES].mean()
display(annual.round(3))
sensitivity = temporal_sensitivity(wide)
display(sensitivity.round(3))

,share_central,share_state,share_local,share_social_security
time,,,,
2015,55.383,3.417,17.538,23.663
2016,55.008,3.420,17.254,24.319
2017,54.549,3.409,17.598,24.445
2018,54.646,3.445,17.770,24.139
2019,54.106,3.435,18.001,24.458
2020,55.565,3.354,16.722,24.358
2021,55.602,3.386,16.920,24.092
2022,55.803,3.387,17.189,23.621
2023,55.755,3.417,17.455,23.373


,specification,pre,post,share_central,share_state,share_local,share_social_security,central_increases,mean_reallocation
0,Principal,2017–2019,2022–2024,1.148,-0.024,-0.456,-0.668,21,2.057
1,Base amplia,2015–2019,2022–2024,0.844,-0.019,-0.299,-0.526,17,2.010
2,Trienio inicial,2015–2017,2022–2024,0.602,-0.009,-0.129,-0.463,16,2.189
3,Bienios próximos,2018–2019,2022–2023,1.404,-0.038,-0.564,-0.802,24,2.207
4,Bienio reciente,2018–2019,2023–2024,1.095,-0.025,-0.480,-0.591,21,2.066
5,Extremos anuales,2019–2019,2024–2024,1.081,-0.022,-0.644,-0.415,22,1.994
6,Fase aguda,2017–2019,2020–2021,1.150,-0.060,-0.969,-0.122,23,2.077
7,Evolución posterior,2020–2021,2022–2024,-0.002,0.036,0.513,-0.547,10,1.222
8,Referencia precrisis,2015–2016,2018–2019,-0.820,0.022,0.490,0.308,3,1.590


El aumento central de 1,148 pp baja a 0,844 con base 2015–2019 y a 0,602 con 2015–2017. Entre los bienios 2015–2016 y 2018–2019, la variación es −0,820 pp. La tendencia anterior impide afirmar una senda plana. No hay dos trienios precrisis no solapados disponibles: la referencia previa usa bienios y no identifica qué habría ocurrido sin pandemia.

La diferencia entre 2020–2021 y 2022–2024 es prácticamente cero para la media central, pero local se recupera y seguridad social pierde cuota. Hay persistencia del promedio central junto con recomposición interna; no se infiere permanencia estructural.

## 5. Comparabilidad institucional y datos provisionales
Las marcas se conservan incluso cuando no modifican el valor. Los [metadatos de Eurostat](https://webgate.ec.europa.eu/eurostat/cache/metadata/en/gov_10a_exp_esms.htm), apartados 18.5 y 19, documentan además cambios que no siempre aparecen como `b` en el total seleccionado. Se revisan Finlandia (reforma de 2023), Eslovaquia (revisiones y tratamiento de operaciones) e Irlanda (serie retrospectiva de S.1314, no un salto mecánico en 2024). Excluirlos es una sensibilidad, no prueba de que el resto de países esté libre de problemas.

In [6]:
display(audit.groupby(["status", "treatment"]).size().to_frame("observaciones"))
delta = changes(wide)
display(
    pd.DataFrame(
        {
            "todos": delta[SHARES].mean(),
            "sin_FI_SK_IE": delta.drop(["FI", "SK", "IE"])[SHARES].mean(),
            "sin_Malta": delta.drop("MT")[SHARES].mean(),
        }
    ).round(3)
)
grouped = grouped_change(wide)
print("Central + seguridad social:", grouped.mean().iloc[0])
print(
    "Chipre, central y agrupado:",
    delta.loc["CY", "share_central"],
    grouped.loc["CY"].iloc[0],
)

,,observaciones
status,treatment,
,valor_publicado,1031
b,valor_publicado,2
m,cero_estructural_documentado,240
p,valor_publicado,77


,todos,sin_FI_SK_IE,sin_Malta
share_central,1.148,1.090,1.185
share_state,-0.024,-0.027,-0.025
share_local,-0.456,-0.546,-0.466
share_social_security,-0.668,-0.517,-0.694


Central + seguridad social: 0.47999203400043094
Chipre, central y agrupado: -6.320177192837875 0.3040083601722756


## 6. Inferencia y reproducibilidad
Test t de diferencias entre países, bilateral; familia principal de tres cuotas. Holm corrige multiplicidad dentro de esa familia. Wilcoxon se muestra como comprobación distinta, sin elegirlo por significación. Las cuotas están ligadas por una suma constante y los países por shocks comunes; estos hechos no desaparecen por corregir valores p.

El índice de reasignación es no negativo. Su media frente a cero no evalúa si la pandemia produjo una recomposición excepcional: se describe, y se compara con referencias temporales sin asignarle una interpretación causal.

La instantánea se conserva sin descargar una versión nueva y cualquier falta de completitud detiene el cálculo. El PDF usa tablas y macros numéricas generadas por las mismas funciones del cuaderno principal. El manifiesto guarda huellas de entradas, código y salidas; el ZIP incluye la evidencia para ejecutar sin descargar datos.

## Conclusión metodológica
La especificación es defendible para describir una diferencia entre etapas. Los ceros tienen fundamento institucional acotado; los periodos definen una pregunta económica concreta; la serie y las sensibilidades muestran lo que esa simplificación omite. La interpretación final permanece limitada a composición contable y no a causalidad, eficiencia o autonomía.